# 01 — Budowa i testowanie zbioru danych (CycleGAN)

**Projekt:** konwersja koszykówka uliczna ↔ koszykówka profesjonalna (NBA)

Ten notebook:
1. przygotowuje surowe zdjęcia z dwóch domen,
2. buduje ujednolicony, niesparowany zbiór `trainA / trainB / testA / testB`,
3. **testuje** zbiór — sprawdza wymiary, liczność, podgląda próbki.

CycleGAN nie wymaga par — domeny A i B to dwa **niezależne** zbiory zdjęć.

## 0. Konfiguracja Google Colab

**Wykonaj najpierw te kroki:**
1. Włącz GPU: *Runtime → Change runtime type → Hardware accelerator → GPU (T4)*.
2. Uruchom komórkę poniżej — zamontuje Twój Dysk Google.
3. Pliki projektu (notebooki + folder `scripts/`) wgraj na Dysk do folderu `cyclegan_basketball`.

Dzięki montowaniu Dysku dane i wytrenowane modele **nie znikną** po rozłączeniu sesji Colab.

In [ ]:
# --- montowanie Dysku Google ---
from google.colab import drive
drive.mount("/content/drive")

import os
# folder projektu na Twoim Dysku Google (utwórz go i wgraj tam pliki)
PROJEKT = "/content/drive/MyDrive/cyclegan_basketball"
os.makedirs(PROJEKT, exist_ok=True)
os.chdir(PROJEKT)
print("Katalog roboczy:", os.getcwd())
print("Zawartość:", os.listdir("."))

## 1. Konfiguracja

Ustaw ścieżki i parametry. W Google Colab najpierw zamontuj Dysk lub wgraj pliki.

In [ ]:
import os

# katalog główny projektu
ROOT = "."
DATA = os.path.join(ROOT, "data")

# domena A = koszykówka uliczna, domena B = koszykówka profesjonalna
RAW_A = os.path.join(DATA, "raw_street")   # tu wrzuć zdjęcia uliczne
RAW_B = os.path.join(DATA, "raw_pro")      # tu wrzuć zdjęcia z hal/NBA

IMG_SIZE = 256        # docelowy rozmiar (256 zalecane na GPU, 128 na słabszym sprzęcie)
TEST_RATIO = 0.1      # 10% zdjęć trafia do zbioru testowego

for d in [RAW_A, RAW_B]:
    os.makedirs(d, exist_ok=True)
print("Katalogi gotowe. Wrzuć zdjęcia do:")
print(" ", RAW_A, " <- koszykówka uliczna")
print(" ", RAW_B, " <- koszykówka profesjonalna (NBA)")

## 1b. Funkcje budowy zbioru (wbudowane)

Poniższe funkcje przygotowują zbiór bezpośrednio w notebooku — dzięki temu nie musisz
osobno wgrywać folderu `scripts/` na Dysk. Notebook jest samodzielny.

In [ ]:
import random
from PIL import Image, ImageOps

def center_square(img):
    """Przycina obraz PIL do kwadratu wzdłuż środka (bez zniekształceń)."""
    w, h = img.size
    s = min(w, h)
    return img.crop(((w-s)//2, (h-s)//2, (w-s)//2+s, (h-s)//2+s))

def process_domain(raw_dir, train_dir, test_dir, size, test_ratio, seed=42):
    """Skaluje zdjęcia jednej domeny i dzieli je na train/test."""
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)
    files = sorted(f for f in os.listdir(raw_dir)
                   if f.lower().endswith((".jpg",".jpeg",".png",".bmp",".webp")))
    if not files:
        print(f"  UWAGA: brak zdjęć w {raw_dir}")
        return 0, 0
    random.Random(seed).shuffle(files)
    n_test = max(1, int(len(files) * test_ratio))
    test_set = set(files[:n_test])
    n_tr, n_te = 0, 0
    for i, fn in enumerate(files):
        try:
            img = Image.open(os.path.join(raw_dir, fn))
            img = ImageOps.exif_transpose(img).convert("RGB")   # naprawa orientacji EXIF
        except Exception as e:
            print(f"  pomijam {fn}: {e}");  continue
        img = center_square(img).resize((size, size), Image.LANCZOS)
        dst = test_dir if fn in test_set else train_dir
        img.save(os.path.join(dst, f"{i:05d}.jpg"), quality=95)
        if fn in test_set: n_te += 1
        else:              n_tr += 1
    return n_tr, n_te

def build_dataset(root="data", size=256, test_ratio=0.1):
    """Buduje pełny zbiór CycleGAN z folderów raw_street/ i raw_pro/."""
    a = process_domain(f"{root}/raw_street", f"{root}/trainA", f"{root}/testA", size, test_ratio)
    b = process_domain(f"{root}/raw_pro",    f"{root}/trainB", f"{root}/testB", size, test_ratio)
    print(f"Domena A (uliczna): trainA={a[0]}  testA={a[1]}")
    print(f"Domena B (pro):     trainB={b[0]}  testB={b[1]}")
    if min(a[0], b[0]) < 100:
        print("UWAGA: mało danych (<100/domenę). Dodaj więcej zdjęć.")
    return a, b

print("Funkcje gotowe.")

## 2. (Opcjonalnie) Dane demonstracyjne

Jeśli nie masz jeszcze prawdziwych zdjęć, możesz wygenerować **syntetyczne**
obrazy demo, aby przetestować cały pipeline. 

> **Do oddania zadania użyj prawdziwych zdjęć!** Skąd je wziąć — patrz `README.md` sekcja 2.
> Wikimedia Commons, Kaggle, Unsplash/Pexels (licencje CC) lub własne zdjęcia boisk.

Uruchom poniższą komórkę tylko jeśli chcesz przetestować kod na danych syntetycznych.

In [ ]:
# --- OPCJONALNIE: syntetyczne dane demo do testu pipeline'u ---
# Do oddania zadania użyj PRAWDZIWYCH zdjęć (patrz README sekcja 2).
# Odkomentuj poniższe, by wygenerować dane demo:
#
# from PIL import ImageDraw, ImageFilter
# rnd = random.Random(123)
# os.makedirs("data/raw_street", exist_ok=True)
# os.makedirs("data/raw_pro", exist_ok=True)
# for i in range(150):
#     # uproszczone obrazy demo - patrz scripts/make_demo_images.py po pełną wersję
#     Image.new("RGB",(256,256),(110,110,115)).save(f"data/raw_street/d_{i:04d}.jpg")
#     Image.new("RGB",(256,256),(200,160,90)).save(f"data/raw_pro/d_{i:04d}.jpg")
# print("Wygenerowano dane demo.")

print("Pomiń, jeśli masz własne zdjęcia w data/raw_street/ i data/raw_pro/")

## 3. Budowa zbioru

Skrypt `build_dataset.py`:
- wczytuje zdjęcia, naprawia orientację EXIF,
- przycina każde do **kwadratu** (środek) — bez zniekształceń proporcji,
- skaluje do `IMG_SIZE × IMG_SIZE`,
- dzieli na zbiór treningowy i testowy.

In [ ]:
# budowa zbioru — przycięcie do kwadratu, skalowanie, podział train/test
build_dataset(root=DATA, size=IMG_SIZE, test_ratio=TEST_RATIO)

## 4. Testowanie zbioru — statystyki

Sprawdzamy, ile obrazów trafiło do każdego podzbioru i czy wszystkie mają poprawny wymiar.

In [ ]:
from PIL import Image

def opisz(folder):
    if not os.path.isdir(folder):
        print(f"{folder}: BRAK")
        return
    pliki = [f for f in os.listdir(folder) if f.lower().endswith((".jpg", ".png"))]
    if not pliki:
        print(f"{folder}: pusty")
        return
    rozmiary = set()
    for f in pliki[:50]:
        with Image.open(os.path.join(folder, f)) as im:
            rozmiary.add(im.size)
    print(f"{folder:20s}: {len(pliki):4d} obrazów, wymiary: {rozmiary}")

for sub in ["trainA", "trainB", "testA", "testB"]:
    opisz(os.path.join(DATA, sub))

## 5. Podgląd próbek

Wizualna kontrola — czy domeny faktycznie się różnią (faktura, kolor, oświetlenie).

In [ ]:
import matplotlib.pyplot as plt
import random

def pokaz_probki(folder, tytul, n=6):
    pliki = sorted(os.listdir(folder))
    pliki = random.sample(pliki, min(n, len(pliki)))
    fig, axes = plt.subplots(1, len(pliki), figsize=(2.4*len(pliki), 2.6))
    if len(pliki) == 1:
        axes = [axes]
    for ax, f in zip(axes, pliki):
        ax.imshow(Image.open(os.path.join(folder, f)))
        ax.axis("off")
    fig.suptitle(tytul, fontsize=13)
    plt.tight_layout()
    plt.show()

pokaz_probki(os.path.join(DATA, "trainA"), "Domena A — koszykówka uliczna")
pokaz_probki(os.path.join(DATA, "trainB"), "Domena B — koszykówka profesjonalna (NBA)")

## 6. Podsumowanie

Zbiór jest gotowy, jeśli:
- każda domena ma sensowną liczbę obrazów (zalecane ≥ 200, minimum ~100),
- wszystkie obrazy mają identyczny wymiar `IMG_SIZE × IMG_SIZE`,
- domeny wizualnie się od siebie różnią.

➡️ Przejdź do **`02_train_cyclegan.ipynb`**, aby wytrenować sieć.